In [1]:
import sys
import os
import numpy as np
from typing import Tuple, List


In [2]:
# 将父目录（项目根目录）添加到 sys.path
# 使用绝对路径更安全，避免路径解析错误
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# 现在可以正常导入 common.util 了
from common.util import preprocess, convert_one_hot

In [3]:
def create_contexts_target(corpus: List[int], window_size: int = 1) -> Tuple[np.ndarray, np.ndarray]:
    """
    根据给定的语料库和窗口大小，生成上下文(contexts)和目标词(target)。

    Args:
        corpus (List[int]): 单词ID列表，代表整个语料库
        window_size (int): 上下文的窗口大小（例如，1表示目标词左右各1个单词）

    Returns:
        Tuple[np.ndarray, np.ndarray]:
        - contexts: 上下文数组，形状为 (m, window_size * 2)，m为样本数
        - target: 目标词数组，形状为 (m,)
    """
    contexts = []
    target = []
    for i in range(len(corpus)):
        if i - window_size < 0 or i + window_size > len(corpus) - 1:
            continue

        cs = []
        for j in range(1, window_size + 1):
            cs.append(corpus[i - j])
            cs.append(corpus[i + j])

        contexts.append(cs)
        target.append(corpus[i])

    return np.array(contexts), np.array(target)

In [ ]:
def cbow_forward(contexts: np.ndarray, target: np.ndarray, W_in: np.ndarray, W_out: np.ndarray) -> float:
    """
    实现 CBOW 模型的正向传播，计算得分和损失。

    Args:
        contexts (np.ndarray): 上下文数据的 one-hot 表示，形状例如 (batch_size, 2, vocab_size)
        target (np.ndarray): 目标词数据的 one-hot 表示，形状例如 (batch_size, vocab_size)
        W_in (np.ndarray): 输入侧的权重矩阵，形状为 (vocab_size, hidden_size)
        W_out (np.ndarray): 输出侧的权重矩阵，形状为 (hidden_size, vocab_size)

    Returns:
        float: 交叉熵损失 (此处可暂不实现完整的 loss 计算，只需算出中间层 h 和最终的 score 即可)
    """
    # 提示：书中假定上下文有两个单词，所以用了 contexts[:, 0] 和 contexts[:, 1]。
    # 如果我们要支持任意多个上下文单词，可以使用 np.mean() 来求平均。

    # 1. 提取所有上下文单词的向量
    # 结果形状变为: (batch_size, n_contexts, hidden_size)
    context_vecs = np.dot(contexts, W_in)

    # 2. 沿着 axis=1 (上下文单词维度) 求平均，得到中间层 h
    # 结果形状变为: (batch_size, hidden_size)
    h = np.mean(context_vecs, axis=1)

    # 3. 将中间层 h 乘以 W_out，得到最终得分 score
    # 结果形状变为: (batch_size, vocab_size)
    score = np.dot(h, W_out)

    return score

In [ ]:
text = 'The best performing models also connect the encoder and decoder through an attention mechanism.'
corpus, word_to_id, id_to_word = preprocess(text)

In [10]:
contexts, target = create_contexts_target(corpus, 1)

vocab_size = len(word_to_id)
contexts = convert_one_hot(contexts, vocab_size)
target = convert_one_hot(target, vocab_size)

In [ ]:
# 假设隐藏层的 size 为 3
vocab_size = 3
W_in = 0.01 * np.random.randn(contexts.shape[2], vocab_size)
W_out = 0.01 * np.random.randn(3, contexts.shape[2])

In [12]:
contexts.shape

(13, 2, 14)

In [13]:
target.shape

(13, 14)